In [ ]:
# Importación de librerías
import pandas as pd
import os
import psycopg2

In [ ]:
# Construcción de rutas de archivos
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

keys_path = os.path.join(base_dir, "keys.txt")
info_videos = os.path.join(base_dir, "data", "info_videos_clean.csv")
info_canal = os.path.join(base_dir, "data", "info_canales_clean.csv")

In [ ]:
# Lectura de los archivos CSV con Pandas
video = pd.read_csv(info_videos, encoding='utf-8-sig')
canal = pd.read_csv(info_canal, encoding='utf-8-sig')

In [ ]:
# Renombrar columnas de los DataFrames para que coincidan con el snake_case de la BD
video = video.rename(columns={'timestamp': 'tiempo_extraccion', 'channelTitle': 'channel_title','publishedAt': 'published_at', 'viewCount': 'view_count','likeCount': 'like_count', 'commentCount': 'comment_count'})

canal = canal.rename(columns={'timestamp_canal': 'tiempo_extraccion','title': 'channel_title','subscriberCount': 'subscriber_count','videoCount': 'video_count','viewCount': 'view_count'})

In [ ]:
# Lectura del archivo de configuración keys.txt
config = {}
with open(keys_path, "r") as file:
    for linea in file:
        linea = linea.strip()
        if not linea or linea.startswith('#'): 
            continue
        if '=' in linea:
            clave, valor = linea.split('=', 1)
            config[clave.strip()] = valor.strip()

HOST = config.get('HOST')
PORT = config.get('PORT')
DATABASE = config.get('DATABASE')
DB_USER = config.get('DB_USER')
DB_PASSWORD = config.get('DB_PASSWORD')

In [ ]:
# Conexión a PostgreSQL
conn = psycopg2.connect(
    host = HOST,
    port = PORT,
    database = DATABASE,
    user = DB_USER,
    password = DB_PASSWORD
)
cur = conn.cursor()

In [ ]:
# Transformación del DataFrame video a lista de tuplas
# Paso 1: convertir el DataFrame a un array de NumPy
array_videos = video.to_numpy()

# Paso 2: crear una lista vacía donde guardaremos las tuplas
datos_videos = []

# Paso 3: recorrer cada fila del array
for fila in array_videos:
    # Paso 4: convertir la fila (que es como una lista) en una tupla
    tupla_fila = tuple(fila)
    # Paso 5: agregar esa tupla a la lista
    datos_videos.append(tupla_fila)

# Preparación e inserción de datos en la tabla videos
insert_videos = """
    INSERT INTO videos 
    (tiempo_extraccion, channel_title, id, published_at, title, view_count, like_count, comment_count, duration_seconds, hora, dia_semana, nombre_dia, engagement_rate, tipo_video)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (id) DO UPDATE SET
        view_count = EXCLUDED.view_count,
        like_count = EXCLUDED.like_count,
        comment_count = EXCLUDED.comment_count,
        engagement_rate = EXCLUDED.engagement_rate,
        tipo_video = EXCLUDED.tipo_video;
"""

# 3. Ejecutar la inserción para todas las filas
cur.executemany(insert_videos, datos_videos)

# 4. Confirmar los cambios
conn.commit()

In [ ]:
# Transformación del DataFrame canal a lista de tuplas
array_canal = canal.to_numpy()

datos_canal = []

for fila_canal in array_canal:
    tupla_fila_canal = tuple(fila_canal)
    datos_canal.append(tupla_fila_canal)

# Inserción en la tabla canal_snapshots
insert_canales = """
    INSERT INTO canal_snapshots 
    (tiempo_extraccion, channel_title, subscriber_count, video_count, view_count)
    VALUES (%s, %s, %s, %s, %s)
    ON CONFLICT (tiempo_extraccion, channel_title) DO NOTHING;
"""
cur.executemany(insert_canales, datos_canal)
conn.commit()

In [ ]:
# Cierre de la conexión
cur.close()
conn.close()